# Memory Store

In [ ]:
from langgraph.store.memory import InMemoryStore

In [ ]:
# create a store
store = InMemoryStore()

In [ ]:
# creating a namespace
namespace = {"user", "u1"}

## Creating Memories

In [ ]:
# Adding memories
store.put(namespace, "1", {"data": "user likes pizza"})
store.put(namespace, "2", {"data": "User prefers dark mode."})

In [ ]:
# another namespace
namespace2 = {"user", "u2"}

# add memories
store.put(namespace2, "1", {"data": "User linkes pasta"})
store.put(namespace2, "2", {"data": "User prefers grid style navigation"})

## Retrieving Memories

In [ ]:
# store.get(namespace,key)
store.get(namespace, "1")

In [ ]:
store.get(namespace2, "1")

In [ ]:
store.get(namespace, "2")

In [ ]:
store.get(namespace2, "2")

## Retrieving all Memories

In [ ]:
items = store.search(namespace)

for item in items:
    print(item.value)

In [ ]:
items = store.search(namespace2)

for item in items:
    print(item.value)

## Semantic Search

In [ ]:
from langchain_openai import OpenAIEmbeddings
from dotenv import load_dotenv

load_dotenv()

embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

In [ ]:
store = InMemoryStore(index={'embed': embedding_model, 'dims': 1536})

In [ ]:
namespace = {"users": "u1"}

In [ ]:
store.put(namespace, "1", {"data": "User prefers concise answers over long expalnation."})
store.put(namespace, "2", {"data": "User likes examples in Python."})
store.put(namespace, "3", {"data": "User usually works late at night."})
store.put(namespace, "4", {"data": "User prefers dark mode in applications."})
store.put(namespace, "5", {"data": "User is learning machine learning"})
store.put(namespace, "6", {"data": "User dislikes overly theoritical explanations."})
store.put(namespace, "7", {"data": "User prefers step-by-step reasoning."})
store.put(namespace, "8", {"data": "User is based in India"})
store.put(namespace, "9", {"data": "User likes real-world analogies."})
store.put(namespace, "10", {"data": "User prefers bullet points over paragraphs."})

In [ ]:
items = store.search(namespace, query = "What is the user currently learning", limit=1)

for item in items:
    print(item.value)

In [ ]:
items = store.search(namespace, query = "What are user preferences", limit=1)

for item in items:
    print(item.value)

# LangGraph Workflow for Memory Store

## Chatbot Reading Existing Memory

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from langchain_openai import ChatOpenAI
from langchain_core,runnables import RunnableConfig
from langchain_core.messages import SystemMessage0

from langgraph.graph import StateGraph, START, END, MessageState
from langgraph.store.memory import InMemoryStore00
from langgraph.store.base import BaseStore

In [ ]:
# ----------------------------------
# 1. Create LTM Store + seed memories (done BEFORE running the graph)
# ----------------------------------

store = InMemoryStore()

user_id = "u1"

# store user details as a single blob (simple for teaching)
# You can also split into multiple records; this keeps it easy  
user_details = {"user", user_id, "details"}

store.put(user_details, "profile_1", {"data": "Name: Koyel"})
store.put(user_details, "profile_2", {"data": "Profession: Teaches AI on YouTube"})
store.put(user_details, "preference_1", {"data": "Perfers concise answers"})
store.put(user_details, "preference_2", {"data": "Likes example in Python"})
store.put(user_details, "project_1", {"data": "Building MCP servers (Python-based project)"})

In [ ]:
# ----------------------------------
# 2. System Prompt Template (My Prompt)
# ----------------------------------
SYSTEM_PROMPT_TEMPLATE = """You are a helpful assistant with memory capabilities.
If user-specific memory is available, use it to personalize your responses based on what you know about the user.

Your goal is to provide relevant, friendly and tailored assistance that reflects the user's preferences, context and past interactions.

If the user's name or relevant personal sontext is avilable, always personalize your responses by:
- Always address the user by name (e.g., "sure, Koyel...") when appropriate 
- Reflecting known projects, tools or preferences (e.g., "Your MCP server python based project")
- Adjusting the tone to feel friendly, natural and diretcly aimed at the user.

Avoid generic phrasing when personalization is possible. For example, instead of "In TypeScript apps...." say "Since your project is built with TypeScript..."

Use personalization especially in:
- Greetings and transitions
- Help or guidance tailored to tools and frameworks the user uses
- Follow-up messages that continue frp, past context

Always ensure that personalization is based only on known user details and not assumed.

In the end suggest 3 relevant further questions based on the current response and user profile

The user's memory (which may be empty) is provided as: {user_details_content}
"""

In [ ]:
# -----------------------------------------------------
# 3. Build Graph: START -> chat -> END (read-only LTM)
# -----------------------------------------------------
llm = ChatOpenAI(model="gpt-4o-mini")


In [ ]:
def chat_node(state: MessageState, config: RunnableConfig, store: BaseStore):
    user_id = config["configurable"]["user_id"]

    # Read-only: fetch user details memory (no writes)

user_details = ("user", user_id, "details")
items = store.search(user_details)

# Convert memory items into a string blob for {user_details_content}
# Keep it dead simple for teaching
if items:
    user_details_content = "\n".join(f"- {it.value.get('data', '')}" for it in items)
else:
    user_details_content = "" # prompt says it may be empty

system_prompt = SYSTEM_PROMPT_TEMPLATE.format(
    user_details_content = user_details_content
)

system_msg = SystemMessage(content=system_prompt)

response = llm.invoke([system_msg] + state["message"])
return {"messages": [response]}

In [ ]:
builder = StateGraph(MessagesState)

builder.add_node("chat", chat_node)
builder.add_edge(START, "chat")
builder.add_edge("chat", END)

graph = builder.compile(store=store)

graph

In [ ]:
# --------------------------------
# 4. Run it (provide user_id in config)
# --------------------------------

config = {"configurable": {"user_id": "u1"}}


result = graph.invoke(
    {"messages": [{"role": "user", "content": "explain gen ai in simple terms."}]}
)

print(result["messages"][-1].content)

## Chatbot Creating of New Memory